# Runs timeresolved across all subjs 

In [1]:
## SETUP 

from pathlib import Path
import numpy as np
import pandas as pd
import mne
from specparam import SpectralGroupModel

psg_dir   = Path("/Users/elizabethkaplan/Desktop/MASS_PSG")
data_dir  = Path("/Users/elizabethkaplan/Desktop/SS2_Data")
save_dir  = Path("/Users/elizabethkaplan/Desktop/SS2_Results/time_resolved")
save_dir.mkdir(exist_ok=True)

#subj excluded for poor quality 
EXCLUDE   = set()   # 01-02-0019 is now included (n=19); retained after preprocessing

subject_ids = [f"01-02-{i:04d}" for i in range(1, 20)]
subject_ids = [s for s in subject_ids if s not in EXCLUDE]

# Analysis parameters
win_sec    = 2.0
step_sec   = 0.50
fmin, fmax = 1.0, 45.0
r2_thresh  = 0.90

stage_order = {
    "Sleep stage W": 0, "Sleep stage 1": 1, "Sleep stage 2": 2,
    "Sleep stage 3": 3, "Sleep stage 4": 3,
    "Sleep stage R": 4, "Sleep stage ?": np.nan,
}

print(f"Subjects to process: {len(subject_ids)}")

Subjects to process: 18


In [2]:
## process all subjs 

all_subjects = {}
failed       = []

for subj_id in subject_ids:

    # Checkpoint: skip if already processed 
    save_path = save_dir / f"{subj_id}_time_resolved.npz"
    if save_path.exists():
        print(f"[SKIP] {subj_id}: already processed, loading from disk")
        d = np.load(save_path, allow_pickle=True)
        all_subjects[subj_id] = {k: d[k] for k in d.files}
        # fix string arrays 
        all_subjects[subj_id]["window_stages"] = list(all_subjects[subj_id]["window_stages"])
        continue

    #paths to psg and expert annotation files 
    psg_path     = psg_dir  / f"{subj_id} PSG.edf"
    kc_path      = data_dir / f"{subj_id} KComplexes_E1.edf"
    spindle_path = data_dir / f"{subj_id} Spindles_E1.edf"
    staging_path = data_dir / f"{subj_id} Base.edf"

    #if subj doesn't have all of the required files, skip it 
    if not all(p.exists() for p in [psg_path, kc_path, spindle_path, staging_path]):
        print(f"[SKIP] {subj_id}: missing file")
        failed.append((subj_id, "missing file"))
        continue

    try:
        print(f"\nProcessing {subj_id}...")
        
        #actually read in the daata from definitions above  
        raw      = mne.io.read_raw_edf(psg_path, preload=True, verbose=False)
        annot    = mne.read_annotations(kc_path)
        spindles = mne.read_annotations(spindle_path)
        stages   = mne.read_annotations(staging_path)
        sfreq    = raw.info["sfreq"]

        #time-resolve C3 channel only 
        c3_idx = [i for i, ch in enumerate(raw.ch_names) if "C3" in ch.upper()]
        c3_data = raw.get_data(picks=c3_idx)

        #convert annotations to array 
        kc_onsets      = np.array([a["onset"] for a in annot])
        spindle_onsets = np.array([a["onset"] for a in spindles])

        # more time-resolve parameters 
        nperseg  = int(win_sec  * sfreq)
        step_smp = int(step_sec * sfreq)
        n_samp   = c3_data.shape[1]
        starts   = np.arange(0, n_samp - nperseg + 1, step_smp)

        #based on window and step size, extract out windows for PSD fit 
        windows = np.stack([c3_data[0, s:s + nperseg].astype(float) for s in starts])

        #fit PSD save power (yaxis: psds_mt) per freq (xaxis: freqs_Tf)
        psds_mt, freqs_tf = mne.time_frequency.psd_array_multitaper(
            windows, sfreq=sfreq, fmin=fmin, fmax=fmax,
            bandwidth=4.0, verbose=False,
        )

        #figure out which sleep stage each window asigned to based on annotations 
        t_centers = (starts + nperseg / 2) / sfreq
        t_hours   = t_centers / 3600.0
     
        window_stages = []
        for t in t_centers:
            label = "Sleep stage ?"
            for ann in stages:
                if ann["onset"] <= t < ann["onset"] + ann["duration"]:
                    label = ann["description"].strip()
                    break
            window_stages.append(label)
        stage_numeric = np.array([stage_order.get(s, np.nan) for s in window_stages])

        #fit spec param across windows 
        fg = SpectralGroupModel(
            peak_width_limits=[1, 12], max_n_peaks=8,
            min_peak_height=0.0, peak_threshold=2.0,
            aperiodic_mode="fixed", verbose=False,
        )
        fg.fit(freqs_tf, np.clip(psds_mt, 1e-30, None), freq_range=[1, 45])

        #store aperiodic params and fit metrics 
        exponent = np.array([r.aperiodic_fit[1] for r in fg.results.group_results])
        offset   = np.array([r.aperiodic_fit[0] for r in fg.results.group_results])
        r2       = np.array([r.metrics.get("gof_rsquared", np.nan) for r in fg.results.group_results])

        good           = r2 >= r2_thresh
        exponent_clean = np.where(good, exponent, np.nan)
        offset_clean   = np.where(good, offset,   np.nan)

        #create df of subject values  
        subj_data = {
            "t_centers":      t_centers,
            "t_hours":        t_hours,
            "exponent":       exponent,
            "offset":         offset,
            "exponent_clean": exponent_clean,
            "offset_clean":   offset_clean,
            "r2":             r2,
            "good":           good,
            "window_stages":  np.array(window_stages),
            "stage_numeric":  stage_numeric,
            "kc_onsets":      kc_onsets,
            "spindle_onsets": spindle_onsets,
            "sfreq":          np.array(sfreq),
            "freqs_tf":       freqs_tf,
        }

        # save time-resolved for each subj as compressed 
        np.savez(save_path, **subj_data)
        print(f"  Saved to {save_path.name}")

        all_subjects[subj_id] = subj_data

        print(f"  {subj_id}: {len(starts)} windows, "
              f"good={good.sum()} ({100*good.mean():.1f}%), "
              f"KCs={len(kc_onsets)}, spindles={len(spindle_onsets)}")

    except Exception as e:
        import traceback
        print(f"[FAIL] {subj_id}: {e}")
        traceback.print_exc()
        failed.append((subj_id, str(e)))

print(f"\nDone: {len(all_subjects)} subjects, {len(failed)} failed")

[SKIP] 01-02-0001: already processed, loading from disk
[SKIP] 01-02-0002: already processed, loading from disk
[SKIP] 01-02-0003: already processed, loading from disk
[SKIP] 01-02-0004: already processed, loading from disk
[SKIP] 01-02-0005: already processed, loading from disk
[SKIP] 01-02-0006: already processed, loading from disk
[SKIP] 01-02-0007: already processed, loading from disk
[SKIP] 01-02-0008: already processed, loading from disk
[SKIP] 01-02-0009: already processed, loading from disk
[SKIP] 01-02-0010: already processed, loading from disk
[SKIP] 01-02-0011: already processed, loading from disk
[SKIP] 01-02-0012: already processed, loading from disk
[SKIP] 01-02-0013: already processed, loading from disk
[SKIP] 01-02-0014: already processed, loading from disk
[SKIP] 01-02-0015: already processed, loading from disk
[SKIP] 01-02-0016: already processed, loading from disk
[SKIP] 01-02-0017: already processed, loading from disk
[SKIP] 01-02-0018: already processed, loading fr